In [ ]:
## Importing libraries
import pandas as pd
import torch
from torch.utils.data import DataLoader,TensorDataset
from torch.nn.utils.rnn import pad_sequence
from transformers import BertTokenizer,BertForSequenceClassification,AdamW
import warnings
warnings.filterwarnings('ignore')

In [ ]:
## Loading the training data
data=pd.read_csv('/kaggle/input/evalbot/training.csv')

In [ ]:
## Dimensions of data
data.shape

In [ ]:
## First 5 rows of data
data.head()

In [ ]:
## Basic information of data
data.info()

In [ ]:
## Drop null values
data=data.dropna()

In [ ]:
## Check for null values
data.isnull().sum()

In [ ]:
## Forming pairs of sentences
training_data=[(row['sentence1'],row['sentence2']) for index,row in data.iterrows()]

In [ ]:
## Length of training pairs
len(training_data)

In [ ]:
training_data[0]

In [ ]:
## Load BERT Tokenizer and Model
model_name='bert-base-uncased'
tokenizer=BertTokenizer.from_pretrained(model_name)
model=BertForSequenceClassification.from_pretrained(model_name,num_labels=3)

In [ ]:
## Tokenization
def tokenization(sent1,sent2):
  encoded=tokenizer.encode_plus(
      sent1,sent2,
      add_special_tokens=True,
      padding=True,
      truncation=True,
      return_tensors='pt'
  )
  input_ids=encoded['input_ids']
  attention_masks=encoded['attention_mask']
  return input_ids,attention_masks

input_ids=[]
attention_masks=[]
for sent1,sent2 in training_data:
  ids,masks=tokenization(sent1,sent2)
  input_ids.append(ids[0])
  attention_masks.append(masks[0])

input_ids=pad_sequence(input_ids,batch_first=True)
attention_masks=pad_sequence(attention_masks,batch_first=True)
similarity_tensor=torch.tensor(data['similarity'].values)
dataset=TensorDataset(input_ids,attention_masks,similarity_tensor)
dataloader=DataLoader(dataset,batch_size=32)

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

In [ ]:
## Initialize hyper-parameters
optimizer1=torch.optim.AdamW(model.parameters(),lr=1e-5)
optimizer2=torch.optim.AdamW(model.parameters(),lr=2e-5)
optimizer3=torch.optim.AdamW(model.parameters(),lr=5e-5)
epochs=10

In [ ]:
## Fine-tuning of model with optimizer 1
for epoch in range(epochs):
  model.train()
  total_loss=0
  for batch in dataloader:
    input_ids,attention_masks,labels=batch
    input_ids,attention_masks,labels=input_ids.to(device),attention_masks.to(device),labels.to(device)
    outputs=model(input_ids,attention_mask=attention_masks,labels=labels)
    loss=outputs.loss
    total_loss+=loss.item()
    optimizer1.zero_grad()
    loss.backward()
    optimizer1.step()
  loss=total_loss/len(dataloader)
  print('Epoch : ',epoch+1,'----> Loss : ',loss)

In [ ]:
## Save fine-tuned model
model.save_pretrained('/kaggle/working/fine-tuned-bert1')

In [ ]:
## Fine-tuning of model with optimizer 2
for epoch in range(epochs):
  model.train()
  total_loss=0
  for batch in dataloader:
    input_ids,attention_masks,labels=batch
    input_ids,attention_masks,labels=input_ids.to(device),attention_masks.to(device),labels.to(device)
    outputs=model(input_ids,attention_mask=attention_masks,labels=labels)
    loss=outputs.loss
    total_loss+=loss.item()
    optimizer2.zero_grad()
    loss.backward()
    optimizer2.step()
  loss=total_loss/len(dataloader)
  print('Epoch : ',epoch+1,'----> Loss : ',loss)

In [ ]:
## Save fine-tuned model
model.save_pretrained('/kaggle/working/fine-tuned-bert2')

In [ ]:
## Fine-tuning of model with optimizer 3
for epoch in range(epochs):
  model.train()
  total_loss=0
  for batch in dataloader:
    input_ids,attention_masks,labels=batch
    input_ids,attention_masks,labels=input_ids.to(device),attention_masks.to(device),labels.to(device)
    outputs=model(input_ids,attention_mask=attention_masks,labels=labels)
    loss=outputs.loss
    total_loss+=loss.item()
    optimizer3.zero_grad()
    loss.backward()
    optimizer3.step()
  loss=total_loss/len(dataloader)
  print('Epoch : ',epoch+1,'----> Loss : ',loss)

In [ ]:
## Save fine-tuned model
model.save_pretrained('/kaggle/working/fine-tuned-bert3')